# 02 — Cálculo de indicadores

Une todos los trimestres filtrados y calcula los indicadores que van al tablero.

**Entrada:** `datos/filtrado/ENOE_T*.csv`
**Salida:** 4 CSV en `datos/analisis/`

Los CSV de análisis sí están en el repositorio (son pequeños).
Los archivos filtrados son más pesados — si no los tienes localmente, corre primero `01_filtrado.ipynb`.

In [1]:
import sys
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Corriendo en Colab')
else:
    print('Corriendo en Jupyter local')

Corriendo en Jupyter local


In [2]:
import pandas as pd
import numpy as np
import glob
import os
import gc
import re
from pathlib import Path

## Rutas

In [3]:
if EN_COLAB:
    RUTA_FILTRADO = '/content/drive/MyDrive/ENOE_filtrado'
    RUTA_SALIDA   = '/content/drive/MyDrive/ENOE_analisis'
else:
    RAIZ = Path('..').resolve()
    RUTA_FILTRADO = str(RAIZ / 'datos' / 'filtrado')
    RUTA_SALIDA   = str(RAIZ / 'datos' / 'analisis')

os.makedirs(RUTA_SALIDA, exist_ok=True)
print(f'Filtrado : {RUTA_FILTRADO}')
print(f'Salida   : {RUTA_SALIDA}')

Filtrado : C:\Users\andre\hackODS\LoboEnsambladores\datos\filtrado
Salida   : C:\Users\andre\hackODS\LoboEnsambladores\datos\analisis


## 1. Unir todos los trimestres

Se agrega año y trimestre a partir del nombre del archivo para poder hacer series temporales.

In [4]:
def parsear_nombre(ruta):
    # ENOE_T120.csv -> trim=1, anio=2020
    # ENOE_TSDEMT412.csv -> trim=4, anio=2012  (formato antiguo)
    nombre = os.path.splitext(os.path.basename(ruta))[0].upper()
    for prefijo in ['ENOE_TSDEMT', 'ENOEN_TSDEMT', 'ENOE_T', 'ENOEN_T']:
        if nombre.startswith(prefijo):
            sufijo = nombre[len(prefijo):]
            if re.fullmatch(r'\d{3,4}', sufijo):
                trim = int(sufijo[0])
                anio = int('20' + sufijo[1:])
                if trim in [1,2,3,4] and 2000 <= anio <= 2030:
                    return trim, anio
    return None


archivos = sorted(glob.glob(os.path.join(RUTA_FILTRADO, '*.csv')))
print(f'Trimestres encontrados: {len(archivos)}')

trozos = []
for ruta in archivos:
    info = parsear_nombre(ruta)
    if info is None:
        print(f'  OMITIDO (nombre no reconocido): {os.path.basename(ruta)}')
        continue
    trim, anio = info
    df = pd.read_csv(ruta, encoding='utf-8-sig', low_memory=False)
    df['trimestre'] = trim
    df['anio']      = anio
    df['periodo']   = f'{anio}-T{trim}'
    trozos.append(df)
    print(f'  {os.path.basename(ruta)} -> {anio}-T{trim}  ({len(df):,} filas)')
    del df
    gc.collect()

datos = pd.concat(trozos, ignore_index=True)
del trozos
gc.collect()

print(f'\nTotal de registros: {len(datos):,}')
print(f'Rango de años: {datos["anio"].min()} – {datos["anio"].max()}')

Trimestres encontrados: 80
  ENOE_T105.csv -> 2005-T1  (164,897 filas)
  ENOE_T106.csv -> 2006-T1  (172,098 filas)
  ENOE_T107.csv -> 2007-T1  (173,235 filas)
  ENOE_T108.csv -> 2008-T1  (173,215 filas)
  ENOE_T109.csv -> 2009-T1  (165,651 filas)
  ENOE_T110.csv -> 2010-T1  (166,283 filas)
  ENOE_T111.csv -> 2011-T1  (163,686 filas)
  ENOE_T112.csv -> 2012-T1  (167,133 filas)
  ENOE_T113.csv -> 2013-T1  (163,319 filas)
  ENOE_T114.csv -> 2014-T1  (167,641 filas)
  ENOE_T115.csv -> 2015-T1  (169,242 filas)
  ENOE_T116.csv -> 2016-T1  (167,881 filas)
  ENOE_T117.csv -> 2017-T1  (167,033 filas)
  ENOE_T118.csv -> 2018-T1  (168,166 filas)
  ENOE_T119.csv -> 2019-T1  (176,997 filas)
  ENOE_T120.csv -> 2020-T1  (184,064 filas)
  ENOE_T121.csv -> 2021-T1  (148,991 filas)
  ENOE_T122.csv -> 2022-T1  (177,295 filas)
  ENOE_T123.csv -> 2023-T1  (204,787 filas)
  ENOE_T124.csv -> 2024-T1  (194,261 filas)
  ENOE_T125.csv -> 2025-T1  (190,503 filas)
  ENOE_T205.csv -> 2005-T2  (167,378 filas)
  ENO

## 2. Ingreso por hora

Se usa `ing_x_hrs` cuando está disponible. Si no, se calcula desde ingocup y hrsocup.
Este indicador es clave porque neutraliza las diferencias de jornada entre géneros.

In [5]:
for col in ['ingocup', 'hrsocup', 'ing_x_hrs']:
    if col in datos.columns:
        datos[col] = pd.to_numeric(datos[col], errors='coerce')

if 'ing_x_hrs' in datos.columns:
    datos['ingreso_hora'] = datos['ing_x_hrs'].where(datos['ing_x_hrs'] > 0, other=np.nan)
else:
    horas_mes = datos['hrsocup'] * 4.33
    datos['ingreso_hora'] = (datos['ingocup'] / horas_mes).where(
        (datos['ingocup'] > 0) & (datos['hrsocup'] > 0), other=np.nan
    ).round(2)

print('ingreso_hora — registros válidos:', datos['ingreso_hora'].notna().sum())
print(datos['ingreso_hora'].describe().round(2))

ingreso_hora — registros válidos: 9779640
count    9779640.00
mean          38.49
std           60.13
min            0.00
25%           16.67
50%           26.09
75%           43.75
max        52096.12
Name: ingreso_hora, dtype: float64


## 3. Brecha salarial por sector y año

Se calcula como el % de diferencia del ingreso mediano por hora entre hombres y mujeres.
Positivo = hombres ganan más. La mediana reduce el efecto de valores extremos.

In [6]:
brecha_sector = (
    datos[datos['ingreso_hora'].notna()]
    .groupby(['anio', 'sector', 'sexo'])['ingreso_hora']
    .median().reset_index()
    .rename(columns={'ingreso_hora': 'mediana_ing_hora'})
)

brecha_pivot = brecha_sector.pivot_table(
    index=['anio', 'sector'], columns='sexo', values='mediana_ing_hora'
).reset_index()
brecha_pivot.columns.name = None

if 'Hombre' in brecha_pivot.columns and 'Mujer' in brecha_pivot.columns:
    brecha_pivot['brecha_pp']  = (brecha_pivot['Hombre'] - brecha_pivot['Mujer']).round(2)
    brecha_pivot['brecha_pct'] = ((brecha_pivot['brecha_pp'] / brecha_pivot['Hombre']) * 100).round(1)

brecha_pivot.to_csv(os.path.join(RUTA_SALIDA, 'brecha_sector_anio.csv'), index=False, encoding='utf-8-sig')
print('brecha_sector_anio.csv guardado.')
print(brecha_pivot[brecha_pivot['anio']==2025].to_string(index=False))

brecha_sector_anio.csv guardado.
 anio                                sector   Hombre    Mujer  brecha_pp  brecha_pct
 2025                          Agropecuario 37.50000 40.00000      -2.50        -6.7
 2025                              Comercio 45.83333 40.00000       5.83        12.7
 2025                          Construcción 55.55556 66.22259     -10.67       -19.2
 2025                              Gobierno 67.82946 74.41860      -6.59        -9.7
 2025                  Industria extractiva 72.91667 93.02326     -20.11       -27.6
 2025                           Manufactura 54.16667 46.51163       7.66        14.1
 2025              Restaurantes y hospedaje 50.00000 43.60465       6.40        12.8
 2025                    Servicios diversos 53.33333 50.00000       3.33         6.2
 2025 Servicios financieros y profesionales 58.13953 58.13953       0.00         0.0
 2025                    Servicios sociales 93.02326 84.88372       8.14         8.8
 2025          Transportes y com

## 4. Serie temporal nacional

Ingreso mediano por hora a nivel nacional por año y sexo.
Es la gráfica de tendencia del tablero.

In [7]:
serie_nacional = (
    datos[datos['ingreso_hora'].notna()]
    .groupby(['anio', 'sexo'])['ingreso_hora']
    .median().reset_index()
    .rename(columns={'ingreso_hora': 'mediana_ing_hora'})
)

serie_nacional.to_csv(os.path.join(RUTA_SALIDA, 'serie_nacional.csv'), index=False, encoding='utf-8-sig')
print('serie_nacional.csv guardado.')
print(serie_nacional)

serie_nacional.csv guardado.
    anio    sexo  mediana_ing_hora
0   2005  Hombre         17.857140
1   2005   Mujer         16.611300
2   2006  Hombre         19.379840
3   2006   Mujer         17.500000
4   2007  Hombre         20.222450
5   2007   Mujer         18.750000
6   2008  Hombre         20.833330
7   2008   Mujer         19.512200
8   2009  Hombre         20.833330
9   2009   Mujer         19.933550
10  2010  Hombre         21.311480
11  2010   Mujer         20.000000
12  2011  Hombre         21.818180
13  2011   Mujer         20.341720
14  2012  Hombre         22.361360
15  2012   Mujer         21.167635
16  2013  Hombre         23.529410
17  2013   Mujer         22.146180
18  2014  Hombre         24.117140
19  2014   Mujer         22.601740
20  2015  Hombre         25.000000
21  2015   Mujer         23.686480
22  2016  Hombre         25.839790
23  2016   Mujer         25.000000
24  2017  Hombre         27.272730
25  2017   Mujer         25.000000
26  2018  Hombre         2

## 5. Brecha por nivel educativo y sector (último año)

Responde la pregunta central: ¿desaparece la brecha cuando el nivel educativo es igual?

In [8]:
ultimo_anio = datos['anio'].max()

reciente = datos[
    (datos['anio'] == ultimo_anio) &
    (datos['ingreso_hora'].notna()) &
    (datos['nivel_educ'].notna()) &
    (datos['sector'].notna())
]

brecha_educ = (
    reciente.groupby(['sector', 'nivel_educ', 'sexo'])['ingreso_hora']
    .median().reset_index()
    .rename(columns={'ingreso_hora': 'mediana_ing_hora'})
)

brecha_educ_pivot = brecha_educ.pivot_table(
    index=['sector', 'nivel_educ'], columns='sexo', values='mediana_ing_hora'
).reset_index()
brecha_educ_pivot.columns.name = None

if 'Hombre' in brecha_educ_pivot.columns and 'Mujer' in brecha_educ_pivot.columns:
    brecha_educ_pivot['brecha_pct'] = (
        (brecha_educ_pivot['Hombre'] - brecha_educ_pivot['Mujer'])
        / brecha_educ_pivot['Hombre'] * 100
    ).round(1)

brecha_educ_pivot.to_csv(
    os.path.join(RUTA_SALIDA, f'brecha_educ_sector_{ultimo_anio}.csv'),
    index=False, encoding='utf-8-sig'
)
print(f'brecha_educ_sector_{ultimo_anio}.csv guardado.')

brecha_educ_sector_2025.csv guardado.


## 6. Informalidad como contexto de apoyo

No es el eje central del análisis. Se usa como anotación en la gráfica de brecha por sector
para mostrar que los sectores con mayor desventaja salarial también tienen más informalidad femenina.

In [9]:
if 'formalidad' in datos.columns:
    inf_contexto = (
        datos[(datos['anio']==ultimo_anio) & (datos['sector'].notna()) & (datos['formalidad'].notna())]
        .groupby(['sector', 'sexo', 'formalidad']).size().reset_index(name='n')
    )
    total = inf_contexto.groupby(['sector','sexo'])['n'].sum().reset_index(name='total')
    inf_contexto = inf_contexto.merge(total, on=['sector','sexo'])
    inf_contexto['pct_informalidad'] = (inf_contexto['n'] / inf_contexto['total'] * 100).round(1)
    inf_contexto = inf_contexto[inf_contexto['formalidad']=='Informal'].drop(columns=['formalidad','n','total'])
    inf_contexto.to_csv(os.path.join(RUTA_SALIDA, f'informalidad_contexto_{ultimo_anio}.csv'), index=False, encoding='utf-8-sig')
    print(f'informalidad_contexto_{ultimo_anio}.csv guardado.')

print('\nArchivos generados:')
for f in sorted(os.listdir(RUTA_SALIDA)):
    print(f'  {f}')

informalidad_contexto_2025.csv guardado.

Archivos generados:
  brecha_educ_sector_2020.csv
  brecha_educ_sector_2025.csv
  brecha_ocupacion_anio.csv
  brecha_sector_anio.csv
  informalidad_contexto_2020.csv
  informalidad_contexto_2025.csv
  serie_nacional.csv


# 7. Brecha salarial por ocupacion y año¶
Se calcula como el % de diferencia del ingreso mediano por hora entre hombres y mujeres. Positivo = hombres ganan más. La mediana reduce el efecto de valores extremos.

In [10]:
brecha_ocupacion = (
    datos[datos['ingreso_hora'].notna()]
    .groupby(['anio', 'ocupacion', 'sexo'])['ingreso_hora']
    .median().reset_index()
    .rename(columns={'ingreso_hora': 'mediana_ing_hora'})
)

brecha_pivot = brecha_ocupacion.pivot_table(
    index=['anio', 'ocupacion'], columns='sexo', values='mediana_ing_hora'
).reset_index()
brecha_pivot.columns.name = None

if 'Hombre' in brecha_pivot.columns and 'Mujer' in brecha_pivot.columns:
    brecha_pivot['brecha_pp']  = (brecha_pivot['Hombre'] - brecha_pivot['Mujer']).round(2)
    brecha_pivot['brecha_pct'] = ((brecha_pivot['brecha_pp'] / brecha_pivot['Hombre']) * 100).round(1)

brecha_pivot.to_csv(os.path.join(RUTA_SALIDA, 'brecha_ocupacion_anio.csv'), index=False, encoding='utf-8-sig')
print('brecha_ocupacion_anio.csv guardado.')
print(brecha_pivot[brecha_pivot['anio']==2025].to_string(index=False))

brecha_ocupacion_anio.csv guardado.
 anio                           ocupacion    Hombre      Mujer  brecha_pp  brecha_pct
 2025         Actividades administrativas  53.57143  44.444440       9.13        17.0
 2025               Actividades agrícolas  43.75000  45.219640      -1.47        -3.4
 2025           Comerciantes y vendedores  43.60465  38.759690       4.84        11.1
 2025           Funcionarios y directivos  85.71429  80.000000       5.71         6.7
 2025 Industria extractiva y construcción  50.89901  55.684755      -4.79        -9.4
 2025            Operadores de maquinaria  37.03704  39.130430      -2.09        -5.6
 2025                   Otras ocupaciones  55.55556  58.139530      -2.58        -4.6
 2025           Profesionistas y técnicos 111.62791 103.359170       8.27         7.4
 2025                Servicios personales  50.00000  56.041670      -6.04       -12.1
 2025        Trabajadores de la educación 110.74197  93.023260      17.72        16.0
 2025             

# 8. Brecha sector ocupacion 2025
Se calcula la el porcentaje de la diferencia del ingreso mediano de hombres y mujeres por el sector y ocupación en 2025: Si es positivo los hombres ganan mas

In [11]:
import os
import pandas as pd

ultimo_anio = datos['anio'].max()

# Filtrar datos del último año con valores válidos
reciente = datos[
    (datos['anio'] == ultimo_anio) &
    (datos['ingreso_hora'].notna()) &
    (datos['nivel_educ'].notna()) &
    (datos['sector'].notna()) &
    (datos['ocupacion'].notna())
]

# Calcular mediana de ingreso por sector, ocupación, nivel educativo y sexo
medianas = (
    reciente.groupby(['sector','ocupacion','nivel_educ','sexo'])['ingreso_hora']
    .median().reset_index()
    .rename(columns={'ingreso_hora':'mediana_ing_hora'})
)

# Pivot para tener columnas Hombre/Mujer
pivot = medianas.pivot_table(
    index=['sector','ocupacion','nivel_educ'], 
    columns='sexo', 
    values='mediana_ing_hora'
).reset_index()
pivot.columns.name = None

# Calcular brecha porcentual si existen ambas columnas
if 'Hombre' in pivot.columns and 'Mujer' in pivot.columns:
    pivot['brecha_pct'] = (
        (pivot['Hombre'] - pivot['Mujer']) / pivot['Hombre'] * 100
    ).round(1)

# Guardar CSV
nombre_csv = f'brecha_ingreso_sector_ocupacion_{ultimo_anio}.csv'
pivot.to_csv(os.path.join(RUTA_SALIDA, nombre_csv), index=False, encoding='utf-8-sig')

print(f'{nombre_csv} guardado.')


brecha_ingreso_sector_ocupacion_2025.csv guardado.
